# Phase 1 Project
## Project Overview

For this project will use data cleaning, imputation, analysis, and visualization to generate insights for a business stakeholder.

## Business Understanding

### 1. Project Purpose


### 2. Problem Statement


### 3. Project Scope

#### This project will focus on the following key areas:



## Key Business Questions to Address:

1. What are the primary risk factors associated with aircraft ownership and operation (e.g., accident rates, )?

3. How can these risk factors be quantified or assessed for different aircraft types?

4. Which specific aircraft models exhibit the lowest overall risk profile based on a comprehensive evaluation of these factors?

5. What are the key characteristics of these low-risk aircraft that make them suitable for the company's initial ventures?

6. What actionable insights and recommendations can be provided to the head of the aviation division to guide their aircraft purchasing

   decisions?

## Data Understanding
The data sources for this analysis will be pulled from the National Transportation Safety Board that includes aviation accident data from 1962 to 2023 about civil aviation accidents and selected incidents in the United States and international waters.
The data is contained in a csv file "Aviation_Data.csv"

### 1. Loading the Data with Pandas
In the cell below, we:

  * Import and alias pandas as pd
  * Import and alias numpy as np
  * Import and alias seaborn as sns
  * Import and alias matplotlib.pyplot as plt
  * Set Matplotlib visualizations to display inline in the notebook

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv("data/Aviation_Data.csv",)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/Aviation_Data.csv'

In [ ]:
df = pd.read_csv("Aviation_Data.csv", low_memory=False )
df.head()

### 2. Famializing with the Data
   * Understanding the dimensionality of the dataset
   * Investigating what type of data it contains, and the data types used to store it
   * Discovering how missing values are encoded, and how many there are
   * Getting a feel for what information it does and doesn't contain

In [ ]:
df.shape

In [ ]:
df.info()

From the information above we can deduce that: most of the items are objects and only 5 out of the 30 columns are floats and Event.Date and Publication.Date are objects instead of datetime.

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df.columns = [col.replace(".", "_") for col in df.columns]
df.columns

In [ ]:
df['Aircraft_damage'].value_counts()

In [ ]:
df['Injury_Severity'].value_counts()

In [ ]:
df['Aircraft_Category'].value_counts()

In [ ]:
df['Make'].value_counts()

In [ ]:
df['Model'].value_counts()

In [ ]:
df['Purpose_of_flight'].value_counts()

In [ ]:
df['Weather_Condition'].value_counts()

In [ ]:
df['Broad_phase_of_flight'].value_counts()

In [ ]:
df['Amateur_Built'].value_counts()

In [ ]:
df['Amateur_Built'].unique()

## 3. Data Cleaning and Filtering

In [ ]:
# Filtering out only airplanes 
# The business problem mentions "airplanes for commercial and private enterprises"
# Also we will filter out the Amateur Built for that purpose
df_filtered = df[df['Aircraft_Category'] == 'Airplane'].copy()
df_filtered = df_filtered[df_filtered['Amateur_Built']!= "Yes"]
df_filtered

In [ ]:
df_filtered['Amateur_Built'].value_counts()

In [ ]:
df_filtered['Aircraft_Category'].value_counts()

In [ ]:
# Convert Event_Date to datetime
df_filtered['Event_Date'] = pd.to_datetime(df_filtered['Event_Date'], errors='coerce')

In [ ]:
# deal with duplicates if any
df_filtered.duplicated().value_counts()

#### Further cleaning on key columns
Filling missing values in injury or damage columns is essential before performing calculations on them. 
For categorical data, some of them already have unknown as a unique variable, makes sense in this case and it will prevent errors.

In [ ]:
df_filtered.isna().sum()

In [ ]:
df_filtered['Total_Fatal_Injuries'].describe()

In [ ]:
df_filtered['Total_Serious_Injuries'].describe()

In [ ]:
df_filtered['Total_Minor_Injuries'].describe()

In [ ]:
df_filtered['Total_Uninjured'].describe()

In [ ]:
# Fill missing injury/damage values with 0 or a sensible default
injury_cols = ['Total_Fatal_Injuries', 'Total_Serious_Injuries', 'Total_Minor_Injuries', 'Total_Uninjured']
for col in injury_cols:
    df_filtered[col] = df_filtered[col].fillna(0).astype(int)

In [ ]:
# filling the missing values of the categorical data  with 'unknown'
aircraft_related_cols = ['Injury_Severity', 'Aircraft_damage', 'Make', 'Model', 'Amateur_Built',
       'Weather_Condition', 'Broad_phase_of_flight']

In [ ]:
for cols in aircraft_related_cols:
    df_filtered[cols] = df_filtered[cols].fillna('Unknown')

# 4. Analyze Key Risk Factors
## Accident Rates by Make and Model
This is fundamental. In order to know which aircraft types are involved in more incidents.

In [ ]:
#Investigating the unique counts of make and model
df_filtered['Make'].value_counts().head(20)

In [ ]:
df_filtered['Make'] = df_filtered['Make'].str.title()

In [ ]:
# Asserting that we dont have the same make with different spellings
df_filtered['Make'].value_counts().head(20)

In [ ]:
make_incidents = df_filtered['Make'].value_counts().head(10)
make_incidents

In [ ]:
# A. Accident Rates by Make and Model
# This is fundamental. We need to know which aircraft types are involved in more incidents.
# Note: Without total fleet size/flight hours, this is a count of incidents, not a true rate.
# However, a higher count still implies higher exposure to incidents in this dataset.

print("\n--- A. Top 10 Aircraft Makes by Incident Count ---")
make_incidents = df_filtered['Make'].value_counts().head(10)
print(make_incidents)

plt.figure(figsize=(12, 6))
sns.barplot(x=make_incidents.values, y=make_incidents.index, hue=make_incidents.index, palette='viridis', legend=False)
plt.title('Top 10 Aircraft Makes by Incident Count (Aircraft Category: Airplane)')
plt.xlabel('Number of Incidents')
plt.ylabel('Aircraft Make')
plt.show()

In [ ]:
top_make_10 = make_incidents.index[0]
top_make_10

In [ ]:
top_5_makes = df_filtered['Make'].value_counts().head(5).index
top_5_makes 

In [ ]:
df_filtered['Total_Injuries'] = df_filtered['Total_Fatal_Injuries'] + df_filtered['Total_Serious_Injuries'] + df_filtered['Total_Minor_Injuries']

In [ ]:
df_filtered

In [ ]:
df_filtered['Total_Injuries'] = df_filtered['Total_Fatal_Injuries'] + df_filtered['Total_Serious_Injuries'] + df_filtered['Total_Minor_Injuries']

In [ ]:
total_injuries_by_make = df_filtered.groupby('Make')['Total_Injuries'].sum().nlargest(10)
total_injuries_by_make

In [ ]:
print("\n--- B. Top 10 Aircraft Models by Incident Count (Requires filtering by Make) ---")
# Example: Let's look at models for the top make (e.g., CESSNA)
top_make = make_incidents.index[0] # Get the most frequent make
model_incidents = df_filtered[df_filtered['Make'] == top_make]['Model'].value_counts().head(10)
print(f"Top 10 Models for {top_make}:")
print(model_incidents)

plt.figure(figsize=(12, 6))
sns.barplot(x=model_incidents.values, y=model_incidents.index, hue=make_incidents.index, palette='viridis', legend=False)
plt.title(f'Top 10 Aircraft Models by Incident Count for {top_make} (Aircraft Category: Airplane)')
plt.xlabel('Number of Incidents')
plt.ylabel('Aircraft Model')
plt.show()

In [ ]:
top_make_1 = make_incidents.index[1] # Get the most frequent make
model_incidents = df_filtered[df_filtered['Make'] == top_make_1]['Model'].value_counts().head(10)


plt.figure(figsize=(12, 6))
sns.barplot(x=model_incidents.values, y=model_incidents.index, hue=make_incidents.index, palette='viridis', legend=False)
plt.title(f'Top 10 Aircraft Models by Incident Count for {top_make_1} (Aircraft Category: Airplane)')
plt.xlabel('Number of Incidents')
plt.ylabel('Aircraft Model')
plt.show()

In [ ]:
# To clean all entries with Fatal and group them all together
df_filtered['Injury_Severity'] = df_filtered['Injury_Severity'].apply(lambda x: 'Fatal' if 'Fatal' in str(x) else x)
df_filtered['Injury_Severity'].value_counts()

In [ ]:
# Get the top 10 makes
top_10_makes = df_filtered['Make'].value_counts().head(10).index

# Filter the DataFrame to include only the top 10 makes
df_top_10_makes = df_filtered[df_filtered['Make'].isin(top_10_makes)]

# Create a contingency table of make and injury severity


In [ ]:
make_injury_alternative = df_top_10_makes.groupby(['Make', 'Injury_Severity']).size().unstack(fill_value=0)
display(make_injury_alternative)

In [ ]:
# Create a contingency table of make and injury severity
make_injury = pd.crosstab(df_top_10_makes['Make'], df_top_10_makes['Injury_Severity'])
make_injury

In [ ]:
# Create the stacked bar chart
make_injury.plot(kind='bar', stacked=True, figsize=(12, 8))

plt.title('Injury Severity by Aircraft Make (Top 10)')
plt.xlabel('Aircraft Make')
plt.ylabel('Number of Incidents')
plt.xticks(rotation=45)
plt.legend(title='Injury Severity')
plt.show()

In [ ]:
# Create a combined injury column for easier overall severity assessment
df_filtered['Has_Fatal_Injury'] = df_filtered['Total_Fatal_Injuries'] > 0
df_filtered['Has_Serious_Injury'] = df_filtered['Total_Serious_Injuries'] > 0

In [ ]:
df_filtered.head()

In [ ]:
# Defining a numerical severity score for Aircraft_damage for easier comparison
damage_mapping = {
    'Destroyed': 3,
    'Substantial': 2,
    'Minor': 1,
    'Unknown': 0,}
df_filtered['Damage_Score'] = df_filtered['Aircraft_damage'].map(damage_mapping)

In [ ]:
df_filtered.head()

In [ ]:
# 3. Quantifying Risk Factors for Different Aircraft Types (Make and Model)

print("--- Quantifying Risk Factors for Aircraft Types ---")

# Let's focus on the top N makes/models to avoid excessively long outputs
top_n_makes = 20 # Adjust as needed
top_n_models_per_make = 5 # Adjust as needed

# A. Accident Frequency (Count of Incidents)
# This is the most basic measure of risk: how often is an aircraft type involved in an incident?
print("\n--- A. Incident Frequency by Aircraft Make ---")
make_incident_counts = df_filtered['Make'].value_counts().nlargest(top_n_makes)
print(make_incident_counts)


In [ ]:
df_filtered['Make'].value_counts().head(30)

In [ ]:
df_filtered['Make'] = df_filtered['Make'].apply(lambda x: 'Airbus' if 'Airbus' in str(x) else x)
df_filtered['Make'].value_counts()

In [ ]:
df_filtered['Make'] = df_filtered['Make'].apply(lambda x: 'Air Tractor' if 'Air Tractor' in str(x) else x)
df_filtered['Make'].value_counts()

In [ ]:
df_filtered['Make'] = df_filtered['Make'].apply(lambda x: 'Cirrus' if 'Cirrus' in str(x) else x)
df_filtered['Make'].value_counts()

In [ ]:
df_filtered['Make'] = df_filtered['Make'].apply(lambda x: 'Aviat' if 'Aviat' in str(x) else x)
df_filtered['Make'].value_counts()

In [ ]:
df_filtered['Make'] = df_filtered['Make'].apply(lambda x: 'Aeronca' if 'Aeronca' in str(x) else x)
df_filtered['Make'].value_counts()

In [ ]:
df_filtered['Make'] = df_filtered['Make'].apply(lambda x: 'Cessna' if 'Cessna' in str(x) else x)
df_filtered['Make'].value_counts()

In [ ]:
# Top 10 Aircraft Models by Incident Count which we will filter by Make
# Example: Let's look at models for the top Make(10) (e.g., Cessna, Boeing, Airbus and Aviat)
top_make = make_incidents.index[0] # Get the most frequent make
top_make_4 = make_incidents.index[3] # Boeing
top_make_8 = make_incidents.index[7] # Airbus
top_make_10 = make_incidents.index[9] # Aviat
model_incidents = df_filtered[df_filtered['Make'] == top_make]['Model'].value_counts().head(10)


plt.figure(figsize=(12, 6)
sns.barplot( x=model_incidents.values, y=model_incidents.index, hue=make_incidents.index, palette='viridis',legend=False )
plt.title(f'Top 10 Aircraft Models by Incident Count for {top_make} (Aircraft Category: Airplane)')
plt.xlabel('Number of Incidents')
plt.ylabel('Aircraft Model')
plt.show()

In [ ]:
# B. Accident Frequency by Model (for top Makes)
print(f"\n--- B. Incident Frequency by Aircraft Model (Top {top_n_models_per_make} models per top {top_n_makes} Makes) ---")
model_incident_counts = {}
for make in make_incident_counts.index:
    models_in_make = df_filtered[df_filtered['Make'] == make]['Model'].value_counts().nlargest(top_n_models_per_make)
    if not models_in_make.empty:
        model_incident_counts[make] = models_in_make
        print(f"\nModels for {make}:")
        print(models_in_make)

In [ ]:
# Visualize top Makes by incident count
plt.figure(figsize=(12, 8))
sns.barplot(x=make_incident_counts.values, y=make_incident_counts.index, palette='viridis')
plt.title(f'Top {top_n_makes} Aircraft Makes by Number of Incidents (Airplanes, Non-Amateur)')
plt.xlabel('Number of Incidents')
plt.ylabel('Aircraft Make')
plt.show()

In [ ]:
# C. Severity of Aircraft Damage
# Calculate average damage score and distribution of damage types per aircraft type

print("\n--- C. Average Aircraft Damage Severity by Make (Higher score = more severe) ---")
make_avg_damage = df_filtered.groupby('Make')['Damage_Score'].mean().nlargest(top_n_makes)
print(make_avg_damage)

print(f"\n--- D. Proportion of 'Destroyed' Damage by Make (Top {top_n_makes}) ---")
make_destroyed_prop = df_filtered[df_filtered['Aircraft_damage'] == 'Destroyed'].groupby('Make').size() / df_filtered.groupby('Make').size()
make_destroyed_prop = make_destroyed_prop.fillna(0).nlargest(top_n_makes) * 100 # Convert to percentage
print(make_destroyed_prop)

In [ ]:
# Visualize Proportion of 'Destroyed' Damage by Make
plt.figure(figsize=(12, 8))
sns.barplot(x=make_destroyed_prop.values, y=make_destroyed_prop.index, palette='Reds_d')
plt.title(f'Top {top_n_makes} Aircraft Makes by Proportion of "Destroyed" Damage')
plt.xlabel('Percentage of Incidents with "Destroyed" Damage')
plt.ylabel('Aircraft Make')
plt.show()


In [ ]:
# Top 10 Aircraft Models by Incident Count which we will filter by Make
# Example: Let's look at models for the top Make(10) (e.g., Cessna, Boeing, Airbus and Aviat)
top_make = make_incidents.index[0] # Get the most frequent make
top_make_4 = make_incidents.index[3] # Boeing
top_make_8 = make_incidents.index[7] # Airbus
top_make_10 = make_incidents.index[9] # Aviat
model_incidents = df_filtered[df_filtered['Make'] == top_make]['Model'].value_counts().head(10)


plt.figure(figsize=(12, 6)
sns.barplot( x=model_incidents.values, y=model_incidents.index, hue=make_incidents.index, palette='viridis',legend=False )
plt.title(f'Top 10 Aircraft Models by Incident Count for {top_make} (Aircraft Category: Airplane)')
plt.xlabel('Number of Incidents')
plt.ylabel('Aircraft Model')
plt.show()

In [ ]:
# E. Human Impact (Fatalities and Injuries)
# Calculate average fatalities/injuries and proportion of incidents with injuries

print("\n--- F. Average Fatalities per Incident by Make ---")
make_avg_fatalities = df_filtered[df_filtered['Total_Fatal_Injuries'] > 0].groupby('Make')['Total_Fatal_Injuries'].mean().nlargest(top_n_makes)
# Or, if we want average across *all* incidents involving that make (including those with 0 fatalities)
# make_avg_fatalities = df_filtered.groupby('Make')['Total_Fatal_Injuries'].mean().nlargest(top_n_makes)
print(make_avg_fatalities)

print(f"\n--- G. Proportion of Incidents with Fatalities by Make (Top {top_n_makes}) ---")
make_fatality_prop = df_filtered[df_filtered['Has_Fatal_Injury']].groupby('Make').size() / df_filtered.groupby('Make').size()
make_fatality_prop = make_fatality_prop.fillna(0).nlargest(top_n_makes) * 100
print(make_fatality_prop)

In [ ]:
# Visualize Proportion of Incidents with Fatalities by Make
plt.figure(figsize=(12, 8))
sns.barplot(x=make_fatality_prop.values, y=make_fatality_prop.index, palette='OrRd_d')
plt.title(f'Top {top_n_makes} Aircraft Makes by Proportion of Incidents with Fatalities')
plt.xlabel('Percentage of Incidents with Fatalities')
plt.ylabel('Aircraft Make')
plt.show()